# Week 34, Day 6: Forge: Week 34 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w034/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
print("1. CV", round(1 / np.sqrt(9), 3), " Fano", round(1 / 9, 3))
print("2.", 1 + 0.2 ** 2 * 10)
print("3.", round(1 / ((1 + 49 * 0.2) / 50), 2), "times")

### Exercise: Part A: tuning across a population

`counts[i]` is neuron i's counts (orientations x trials). Write `shuffle_p(c)` as on Day 3 with 300 shuffles, compute it for every neuron, and set `tuned` to a boolean array, True where p is below 0.01. Compare with the hidden truth `is_tuned`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(1962)
ori = np.arange(0, 180, 22.5); n_neurons, n_trials = 20, 30
is_tuned = np.r_[np.ones(16, bool), np.zeros(4, bool)]
prefs = np.r_[np.linspace(0, 180, 16, endpoint=False), np.zeros(4)]
amp = np.where(is_tuned, 12.0, 0.0)
mean_rate = 3 + amp[:, None] * np.exp(2.0 * (np.cos(2 * np.deg2rad(ori[None, :] - prefs[:, None])) - 1))   # neurons x orientations
z = r.normal(size=(3, len(ori), n_trials))                                                                 # shared trial factors
loggain = 0.15 * z[0][None] + 0.25 * (np.cos(2 * np.deg2rad(prefs))[:, None, None] * z[1][None]
                                       + np.sin(2 * np.deg2rad(prefs))[:, None, None] * z[2][None])
counts = r.poisson(mean_rate[:, :, None] * np.exp(loggain - 0.5 * (0.15**2 + 0.25**2)))                   # neurons x orientations x trials

def osi(c):
    m = c.mean(1)
    return abs((m * np.exp(2j * np.deg2rad(ori))).sum()) / m.sum()

def shuffle_p(c, n_shuf=300):
    ...

pvals = np.array([shuffle_p(counts[i]) for i in range(n_neurons)])
tuned = ...
print("OSI:", np.round([osi(counts[i]) for i in range(n_neurons)], 2))
print("p:  ", np.round(pvals, 3))
print(f"{tuned.sum()} neurons tuned; agreement with the truth: {np.mean(tuned == is_tuned):.0%}")

In [ ]:
import numpy as np
def test_osi():
    "osi computes the vector selectivity index from the orientation means"
    c = counts[0]; m = c.mean(1)
    assert abs(osi(c) - abs((m * np.exp(2j * np.deg2rad(ori))).sum()) / m.sum()) < 1e-12

def test_tuned():
    "the shuffle test finds all 16 tuned neurons and flags at most one of the 4 untuned ones"
    assert tuned[:16].all() and tuned[16:].sum() <= 1

test_osi()
test_tuned()
print("All checks passed")

### Exercise: Part B: limited-range correlations

Write `zscore(c)` for one neuron's counts (orientations x trials). Compute `rsc[(i, j)]` for every pair of tuned neurons, the circular difference in preferred orientation (0 to 90 degrees), and the mean correlation of pairs within 30 degrees (`r_by_diff['similar']`) and more than 60 degrees apart (`r_by_diff['orthogonal']`).

In [ ]:
def zscore(c):
    return ...

idx = np.where(tuned)[0]
Z = {i: zscore(counts[i]).ravel() for i in idx}
rsc, dpref = {}, {}
for a_ in range(len(idx)):
    for b_ in range(a_ + 1, len(idx)):
        i, j = idx[a_], idx[b_]
        rsc[(i, j)] = np.corrcoef(Z[i], Z[j])[0, 1]
        d = abs(prefs[i] - prefs[j]) % 180
        dpref[(i, j)] = min(d, 180 - d)
pairs = list(rsc)
r_by_diff = {"similar": np.mean([rsc[p] for p in pairs if dpref[p] <= 30]),
             "orthogonal": ...}
print(f"{len(pairs)} pairs; mean r_sc {np.mean(list(rsc.values())):.3f}")
print(f"preferences within 30 deg: r_sc {r_by_diff['similar']:.3f};  more than 60 deg apart: {r_by_diff['orthogonal']:.3f}")
plt.scatter([dpref[p] for p in pairs], [rsc[p] for p in pairs], s=10)
plt.xlabel("difference in preferred orientation (deg)"); plt.ylabel("noise correlation"); plt.show()

In [ ]:
import numpy as np
def test_z():
    "zscore removes each orientation's mean"
    zs = zscore(counts[0])
    assert np.allclose(zs.mean(1), 0)

def test_range():
    "pairs with preferences within 30 degrees are more correlated than pairs about 90 degrees apart, and the mean correlation is positive"
    assert r_by_diff["similar"] > r_by_diff["orthogonal"] + 0.1 and np.mean(list(rsc.values())) > 0

test_z()
test_range()
print("All checks passed")